In [6]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

In [7]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.strip().str.replace(" ", "_")

print("Initial shape:", df.shape)
print(df.head())
print(df.info())
print("Missing values before cleaning:\n", df.isna().sum())

df = df.drop_duplicates()
target_numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
for col in target_numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()

for col in numeric_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

for col in categorical_cols:
    if col != "customerID":
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(["nan", "None", ""], np.nan)
        
        mode_val = df[col].mode()
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

if "tenure" in df.columns and "MonthlyCharges" in df.columns:
    df["Estimated_Lifetime_Charges"] = df["tenure"] * df["MonthlyCharges"]

numeric_cols = df.select_dtypes(include="number").columns.tolist()

for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr > 0:
        df[col + "_OutlierFlag"] = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
    else:
        df[col + "_OutlierFlag"] = False

print("Cleaned shape:", df.shape)
print("Remaining missing values:\n", df.isna().sum())

df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)
print("Cleaned dataset successfully saved to 'Telecom_Customer_Churn_Cleaned.csv'")

Initial shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingM